## 1. Fixed Window Rate Limiting

In [1]:
import redis
import hashlib
import json
import time
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

# ─────────────────────────────────────────────
#  REDIS CONNECTION
# ─────────────────────────────────────────────
r = redis.Redis(
    host=os.getenv('REDIS_HOST', 'localhost'),
    port=int(os.getenv('REDIS_PORT', 6379)),
    decode_responses=True
)

# ─────────────────────────────────────────────
#  LLM INITIALIZATION (via Evren API)
# ─────────────────────────────────────────────
llm = ChatOpenAI(
    model=os.getenv("EVREN_MODEL"),
    api_key=os.getenv("EVREN_API_KEY"),
    base_url=os.getenv("EVREN_BASE_URL"),
    temperature=0
)

# ─────────────────────────────────────────────
#  HEALTH CHECK
# ─────────────────────────────────────────────
r.ping()

print("\n" + "═" * 58)
print("\033[1;36m  🚀  SYSTEM INITIALIZATION\033[0m")
print("═" * 58)

redis_status = "\033[1;32m✅ CONNECTED\033[0m" if r.ping() else "\033[1;31m❌ FAILED\033[0m"
print(f"  \033[1;33m▸ Redis\033[0m            → {redis_status}")
print(f"     \033[90mhost:\033[0m {os.getenv('REDIS_HOST', 'localhost')}:{os.getenv('REDIS_PORT', 6379)}")

api_status = "\033[1;32m✅ LOADED\033[0m" if os.getenv("EVREN_API_KEY") else "\033[1;31m❌ MISSING\033[0m"
print(f"  \033[1;33m▸ Evren API Key\033[0m    → {api_status}")

model_name = os.getenv("EVREN_MODEL", "unknown")
print(f"  \033[1;33m▸ LLM Model\033[0m        → \033[1;32m{model_name}\033[0m")
print(f"     \033[90mbase_url:\033[0m {os.getenv('EVREN_BASE_URL', 'N/A')}")

print("═" * 58 + "\n")


══════════════════════════════════════════════════════════
  🚀  SYSTEM INITIALIZATION
══════════════════════════════════════════════════════════
  ▸ Redis            → ✅ CONNECTED
     host: localhost:6379
  ▸ Evren API Key    → ✅ LOADED
  ▸ LLM Model        → deepseek-v4-flash
     base_url: https://evren-llmapi.ssyz.org.tr/v1
══════════════════════════════════════════════════════════



In [2]:
class FixedWindowRateLimiter:
    """
    Fixed window rate limiter using Redis INCR.

    Each user gets a counter that resets every `window_seconds`.
    If the counter exceeds `limit`, the request is blocked.
    """

    def __init__(self, redis_client, limit: int = 10, window_seconds: int = 60):
        self.r = redis_client
        self.limit = limit
        self.window = window_seconds

    def check(self, identifier: str) -> dict:
        """
        Check if `identifier` is within rate limit.
        Returns a dict with: allowed, count, remaining, reset_in_seconds
        """
        key = f"rate:fixed:{identifier}"

        # Atomic increment — works safely under concurrent load
        count = self.r.incr(key)

        # On the first request in a window, set the expiry
        if count == 1:
            self.r.expire(key, self.window)

        ttl = self.r.ttl(key)
        remaining = max(0, self.limit - count)
        allowed = count <= self.limit

        if not allowed:
            # Undo the increment — the request was rejected
            self.r.decr(key)

        return {
            "allowed": allowed,
            "count": count if allowed else count - 1,
            "limit": self.limit,
            "remaining": remaining,
            "reset_in_seconds": ttl,
        }

    def reset(self, identifier: str):
        self.r.delete(f"rate:fixed:{identifier}")


# Create a limiter: 5 requests per 30 seconds per user
limiter = FixedWindowRateLimiter(r, limit=5, window_seconds=30)
print("Rate limiter created: 5 requests per 30 seconds")

Rate limiter created: 5 requests per 30 seconds


In [3]:
# Demo: simulate a user making many requests
user_id = "demo_user_42"
limiter.reset(user_id)  # Start fresh

print("\n" + "═" * 72)
print("\033[1;36m  🚦  RATE LIMITER DEMO\033[0m")
print(f"\033[90m  Simulating 8 requests for user '{user_id}' (limit = 5 per 30s)\033[0m")
print("═" * 72)

# ─────────────────────────────────────────────
# TABLE HEADER
# ─────────────────────────────────────────────
print(f"\n  \033[1;33m{'REQUEST':>8}  {'STATUS':>10}  {'COUNT':>6}  "
      f"{'REMAINING':>10}  {'RESET IN':>10}\033[0m")
print("  " + "─" * 62)

# ─────────────────────────────────────────────
# PROCESS EACH REQUEST
# ─────────────────────────────────────────────
for i in range(1, 9):
    result = limiter.check(user_id)

    # Determine color and label based on allowed status
    if result["allowed"]:
        status_label = "\033[1;32m✅ ALLOWED\033[0m"
    else:
        status_label = "\033[1;31m⛔ BLOCKED\033[0m"

    # Color the remaining count based on how close to the limit
    remaining = result["remaining"]
    if remaining <= 0:
        remaining_color = "\033[1;31m"      # red = no quota left
    elif remaining <= 2:
        remaining_color = "\033[1;33m"      # yellow = low
    else:
        remaining_color = "\033[1;32m"      # green = plenty

    print(f"  \033[90m{i:>8}\033[0m  {status_label:<21}  "
          f"\033[1;37m{result['count']:>6}\033[0m  "
          f"{remaining_color}{remaining:>10}\033[0m  "
          f"\033[90m{result['reset_in_seconds']:>9}s\033[0m")

print("  " + "─" * 62)

# ─────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────
print(f"\n  \033[1;34m📊 Summary:\033[0m")
print(f"     \033[1;32m✅ Allowed\033[0m   → \033[1;32m5\033[0m requests")
print(f"     \033[1;31m⛔ Blocked\033[0m   → \033[1;31m3\033[0m requests")
print(f"     \033[90m⏱️  Window\033[0m    → 30 seconds (sliding)")
print(f"     \033[90m🔢 Limit\033[0m     → 5 requests per window")

# Reset after demo
limiter.reset(user_id)
print(f"\n  \033[1;33m🔄 Limiter reset for '{user_id}'\033[0m")

print("\n" + "═" * 72 + "\n")


════════════════════════════════════════════════════════════════════════
  🚦  RATE LIMITER DEMO
  Simulating 8 requests for user 'demo_user_42' (limit = 5 per 30s)
════════════════════════════════════════════════════════════════════════

   REQUEST      STATUS   COUNT   REMAINING    RESET IN
  ──────────────────────────────────────────────────────────────
         1  ✅ ALLOWED        1           4         30s
         2  ✅ ALLOWED        2           3         30s
         3  ✅ ALLOWED        3           2         30s
         4  ✅ ALLOWED        4           1         30s
         5  ✅ ALLOWED        5           0         30s
         6  ⛔ BLOCKED        5           0         30s
         7  ⛔ BLOCKED        5           0         30s
         8  ⛔ BLOCKED        5           0         30s
  ──────────────────────────────────────────────────────────────

  📊 Summary:
     ✅ Allowed   → 5 requests
     ⛔ Blocked   → 3 requests
     ⏱️  Window    → 30 seconds (sliding)
     🔢 Limit     → 5

## Integrating with LLM Calls

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm_limiter = FixedWindowRateLimiter(r, limit=3, window_seconds=60)
def ask_llm_with_rate_limit(user_id: str, prompt: str) -> dict:
    """Ask LLM only if the user is within their rate limit."""
    limit_check = llm_limiter.check(user_id)
    if not limit_check["allowed"]:
        return {
            "success": False,
            "error": "rate_limited",
            "message": f"Rate limit reached. Please wait {limit_check['reset_in_seconds']} seconds.",
            "reset_in_seconds": limit_check["reset_in_seconds"]
        }
    # User is within limit — call your configured LLM
    response = llm.invoke([HumanMessage(content=prompt)])
    return {
        "success": True,
        "response": response.content,
        "requests_used": limit_check["count"],
        "requests_remaining": limit_check["remaining"],
        "reset_in_seconds": limit_check["reset_in_seconds"]
    }
# ─────────────────────────────────────────────
# 3. TEST RUN
# ─────────────────────────────────────────────
user_id = "test_user_99"
llm_limiter.reset(user_id)
prompts = [
    "What is Redis?",
    "What is caching?",
    "What is an LLM?",
    "What is Python?",   # should be blocked
    "What is Docker?",   # should be blocked
]
print("\n" + "═" * 78)
print("\033[1;36m  🚦  LLM RATE LIMITER TEST\033[0m")
print(f"\033[90m  User: '{user_id}' · Limit: 3 requests per 60s\033[0m")
print("═" * 78)
allowed_count = 0
blocked_count = 0
for i, prompt in enumerate(prompts, 1):
    result = ask_llm_with_rate_limit(user_id, prompt)
    print("\n" + "─" * 78)
    print(f"\033[1;35m  📨  Request {i}/{len(prompts)}\033[0m  \033[90m→ {prompt}\033[0m")
    print("─" * 78)
    if result["success"]:
        allowed_count += 1
        used = result['requests_used']
        remaining = result['requests_remaining']
        # Color the remaining count based on quota
        if remaining <= 0:
            remaining_color = "\033[1;31m"      # red = no quota left
        elif remaining <= 1:
            remaining_color = "\033[1;33m"      # yellow = low
        else:
            remaining_color = "\033[1;32m"      # green = plenty
        print(f"\n  \033[1;32m✅ ALLOWED\033[0m")
        print(f"     \033[1;34m▸ Quota\033[0m      → \033[1;37m{used}/3\033[0m used  "
              f"({remaining_color}{remaining} remaining\033[0m)")
        print(f"     \033[1;34m▸ Reset in\033[0m   → \033[90m{result['reset_in_seconds']}s\033[0m")
        print(f"\n     \033[1;34m▸ Response\033[0m")
        print(f"        \033[90m{result['response'][:80]}...\033[0m")
    else:
        blocked_count += 1
        print(f"\n  \033[1;31m⛔ BLOCKED\033[0m")
        print(f"     \033[1;34m▸ Reason\033[0m     → \033[90m{result['error']}\033[0m")
        print(f"     \033[1;34m▸ Message\033[0m    → \033[1;33m{result['message']}\033[0m")
# ─────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────
print("\n" + "═" * 78)
print("\033[1;34m  📊  SUMMARY\033[0m")
print("═" * 78)
total = len(prompts)
hit_rate = allowed_count / total * 100
print(f"\n  \033[1;32m✅ Allowed\033[0m      → \033[1;32m{allowed_count}\033[0m / {total}")
print(f"  \033[1;31m⛔ Blocked\033[0m      → \033[1;31m{blocked_count}\033[0m / {total}")
print(f"  \033[1;34m📈 Pass rate\033[0m    → \033[1;37m{hit_rate:.0f}%\033[0m")
print(f"  \033[90m⏱️  Window\033[0m       → 60 seconds (fixed)")
print(f"  \033[90m🔢 Limit\033[0m        → 3 requests per window")
# Reset after demo
llm_limiter.reset(user_id)
print(f"\n  \033[1;33m🔄 Limiter reset for '{user_id}'\033[0m")
print("\n" + "═" * 78 + "\n")


══════════════════════════════════════════════════════════════════════════════
  🚦  LLM RATE LIMITER TEST
  User: 'test_user_99' · Limit: 3 requests per 60s
══════════════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────────────
  📨  Request 1/5  → What is Redis?
──────────────────────────────────────────────────────────────────────────────

  ✅ ALLOWED
     ▸ Quota      → 1/3 used  (2 remaining)
     ▸ Reset in   → 60s

     ▸ Response
        Redis is an open-source, in-memory data structure store. It is often used as a *...

──────────────────────────────────────────────────────────────────────────────
  📨  Request 2/5  → What is caching?
──────────────────────────────────────────────────────────────────────────────

  ✅ ALLOWED
     ▸ Quota      → 2/3 used  (1 remaining)
     ▸ Reset in   → 57s

     ▸ Response
        Caching is a technique used to store copies of data or files in a temporary s

## Sliding Window Rate Limiting

In [6]:
class SlidingWindowRateLimiter:
    """
    Sliding window rate limiter using Redis Sorted Sets.

    Stores the timestamp of each request as a score in a Sorted Set.
    To check the limit: count requests with score > (now - window_seconds).
    Old timestamps are removed automatically.

    More accurate than fixed window but uses slightly more memory.
    """

    def __init__(self, redis_client, limit: int = 10, window_seconds: int = 60):
        self.r = redis_client
        self.limit = limit
        self.window = window_seconds

    def check(self, identifier: str) -> dict:
        key = f"rate:sliding:{identifier}"
        now = time.time()
        window_start = now - self.window

        # Use a pipeline for atomic execution
        pipe = self.r.pipeline()

        # Remove timestamps older than the window
        pipe.zremrangebyscore(key, 0, window_start)

        # Count current requests in the window
        pipe.zcard(key)

        # Add the current request timestamp
        pipe.zadd(key, {str(now): now})

        # Set TTL to clean up the key
        pipe.expire(key, self.window + 1)

        _, current_count, _, _ = pipe.execute()

        # current_count is the count BEFORE this request
        allowed = current_count < self.limit

        if not allowed:
            # Remove the just-added timestamp since we're rejecting
            self.r.zrem(key, str(now))

        return {
            "allowed": allowed,
            "count": current_count + (1 if allowed else 0),
            "limit": self.limit,
            "remaining": max(0, self.limit - current_count - (1 if allowed else 0)),
            "window_seconds": self.window
        }

    def reset(self, identifier: str):
        self.r.delete(f"rate:sliding:{identifier}")


sliding_limiter = SlidingWindowRateLimiter(r, limit=5, window_seconds=60)

test_id = "sliding_demo_user"
sliding_limiter.reset(test_id)

# ─────────────────────────────────────────────
# OUTPUT — DEMO
# ─────────────────────────────────────────────
print("\n" + "═" * 72)
print("\033[1;36m  🪟  SLIDING WINDOW RATE LIMITER DEMO\033[0m")
print(f"\033[90m  User: '{test_id}' · Limit: {sliding_limiter.limit} per {sliding_limiter.window}s\033[0m")
print("═" * 72)

# ─────────────────────────────────────────────
# TABLE HEADER
# ─────────────────────────────────────────────
print(f"\n  \033[1;33m{'REQUEST':>8}  {'STATUS':<12} {'COUNT':>6}  {'REMAINING':>10}\033[0m")
print("  " + "─" * 46)

# ─────────────────────────────────────────────
# PROCESS EACH REQUEST
# ─────────────────────────────────────────────
allowed_count = 0
blocked_count = 0

for i in range(1, 8):
    result = sliding_limiter.check(test_id)

    # Determine color and label based on allowed status
    if result["allowed"]:
        status_label = "\033[1;32m✅ ALLOWED\033[0m"
        allowed_count += 1
    else:
        status_label = "\033[1;31m⛔ BLOCKED\033[0m"
        blocked_count += 1

    # Color the remaining count based on how close to the limit
    remaining = result["remaining"]
    if remaining <= 0:
        remaining_color = "\033[1;31m"      # red = no quota left
    elif remaining <= 1:
        remaining_color = "\033[1;33m"      # yellow = low
    else:
        remaining_color = "\033[1;32m"      # green = plenty

    print(f"  \033[90m{i:>8}\033[0m  {status_label:<23} "
          f"\033[1;37m{result['count']:>6}\033[0m  "
          f"{remaining_color}{remaining:>10}\033[0m")

print("  " + "─" * 46)

# ─────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────
total = 7
hit_rate = allowed_count / total * 100

print(f"\n  \033[1;34m📊 Summary:\033[0m")
print(f"     \033[1;32m✅ Allowed\033[0m      → \033[1;32m{allowed_count}\033[0m / {total}")
print(f"     \033[1;31m⛔ Blocked\033[0m      → \033[1;31m{blocked_count}\033[0m / {total}")
print(f"     \033[1;34m📈 Pass rate\033[0m    → \033[1;37m{hit_rate:.0f}%\033[0m")
print(f"     \033[90m⏱️  Window\033[0m       → {sliding_limiter.window}s (sliding)")
print(f"     \033[90m🔢 Limit\033[0m        → {sliding_limiter.limit} requests per window")

# Reset after demo
sliding_limiter.reset(test_id)
print(f"\n  \033[1;33m🔄 Limiter reset for '{test_id}'\033[0m")

print("\n" + "═" * 72 + "\n")


════════════════════════════════════════════════════════════════════════
  🪟  SLIDING WINDOW RATE LIMITER DEMO
  User: 'sliding_demo_user' · Limit: 5 per 60s
════════════════════════════════════════════════════════════════════════

   REQUEST  STATUS        COUNT   REMAINING
  ──────────────────────────────────────────────
         1  ✅ ALLOWED         1           4
         2  ✅ ALLOWED         2           3
         3  ✅ ALLOWED         3           2
         4  ✅ ALLOWED         4           1
         5  ✅ ALLOWED         5           0
         6  ⛔ BLOCKED         5           0
         7  ⛔ BLOCKED         5           0
  ──────────────────────────────────────────────

  📊 Summary:
     ✅ Allowed      → 5 / 7
     ⛔ Blocked      → 2 / 7
     📈 Pass rate    → 71%
     ⏱️  Window       → 60s (sliding)
     🔢 Limit        → 5 requests per window

  🔄 Limiter reset for 'sliding_demo_user'

════════════════════════════════════════════════════════════════════════

